In [6]:
import pandas as pd
import numpy as np
from IPython.display import display

In [7]:
from pathlib import Path

DATA_DIR = Path.cwd() / "data" / "nfl-big-data-bowl-2027"
LABELLED_TRACKING_PATH = Path.cwd() / "outputs" / "dl_tracking_labeled.parquet"

if not LABELLED_TRACKING_PATH.exists():
    raise FileNotFoundError(f"{LABELLED_TRACKING_PATH} not found. Run combine_tracking_labeling.ipynb first to create it.")

players = pd.read_parquet(DATA_DIR / "players.parquet")
dl_tracking = pd.read_parquet(LABELLED_TRACKING_PATH)
dl_tracking["time"] = pd.to_datetime(dl_tracking["time"], errors="raise")
print(f"Loaded {len(dl_tracking):,} labeled DL tracking rows across {dl_tracking['nfl_id'].nunique():,} players.")

Loaded 50,570 labeled DL tracking rows across 118 players.


## Working assumptions

Initial drill guesses based on names and the trajectory review in `combine_tracking_labeling.ipynb`: **cutting** uses four-bag agility, pass rush, and body control; **explosion** uses front-start-wave/lateral-reaction, four-bag agility, pass rush, and body control; **bend** focuses on run-the-hoop, with pass rush as a secondary candidate. Review plotted attempts and revise these sets as needed.

This notebook reads the labeled DL tracking dataframe saved by `combine_tracking_labeling.ipynb`; run that notebook first. It does not repeat event labeling. For plants, the source `event_side` label determines left/right; forward/backward events are not direction-scored. Braking time is from the last local speed peak after the prior plant (or attempt start) to the plant; plant dwell is the contiguous period below 1.5 yd/s; the 0.5-second speed gain uses the sample five frames after the plant. Bend retention is minimum bend speed divided by speed entering the bend. Tightest radius is `speed / angular_speed` in yards, only while above 4 yd/s.

Each input is percentile-ranked within its drill (ties receive average rank); lower braking/dwell/recovery-time/radius and higher speed gain/retention are better. The component quality is the mean available percentile (0–1), multiplied by side (`-1` for left, `+1` for right). The sign reports the side and magnitude is relative performance. On a single event, a near-zero value indicates low relative performance; in player aggregates, left/right event scores can cancel, so a value near zero can instead indicate balanced results. These are descriptive, within-dataset scores.

In [8]:
EVENT_GROUP_COLUMNS = ["draft_year", "event_id", "nfl_id", "drill_name", "attempt"]
FRAME_SECONDS = 0.1
PLANT_MAX_SPEED = 1.5
REQUIRED_EVENT_COLUMNS = {
    "event_start", "event_plant_cut", "event_bend", "in_bend",
    "event_side", "event_turn_degrees",
}
missing_event_columns = REQUIRED_EVENT_COLUMNS.difference(dl_tracking.columns)
if missing_event_columns:
    raise ValueError(f"Saved labeled dataframe is missing columns: {sorted(missing_event_columns)}")

def wrap_degrees(angle):
    return (angle + 180) % 360 - 180

print({"plant_cuts": int(dl_tracking["event_plant_cut"].sum()),
       "bends": int(dl_tracking["event_bend"].sum()),
       "drills": sorted(dl_tracking["drill_name"].dropna().unique().tolist())})

{'plant_cuts': 735, 'bends': 852, 'drills': ['BODY_CONTROL_DRILL', 'FOUR_BAG_AGILITY_DRILL', 'FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION', 'PASS_RUSH_DRILL', 'RUN_THE_HOOP_DRILL']}


In [9]:
def plant_event_metrics(attempt, plant_index):
    attempt = attempt.sort_values("time")
    speed = pd.to_numeric(attempt["s"], errors="coerce").to_numpy(dtype=float)
    seconds = (attempt["time"] - attempt["time"].iloc[0]).dt.total_seconds().to_numpy()
    plant_speed = speed[plant_index]
    prior_plants = np.flatnonzero(attempt["event_plant_cut"].to_numpy())
    prior_plants = prior_plants[prior_plants < plant_index]
    interval_start = int(prior_plants[-1] + 1) if len(prior_plants) else 0
    candidates = np.arange(interval_start, plant_index)
    if not len(candidates):
        braking_time = np.nan
    else:
        peaks = [i for i in candidates if i > interval_start and i + 1 < plant_index
                 and speed[i] >= speed[i - 1] and speed[i] >= speed[i + 1]]
        peak_index = peaks[-1] if peaks else int(candidates[np.nanargmax(speed[candidates])])
        braking_time = seconds[plant_index] - seconds[peak_index]

    low_start = plant_index
    low_end = plant_index
    while low_start > 0 and speed[low_start - 1] < PLANT_MAX_SPEED:
        low_start -= 1
    while low_end + 1 < len(speed) and speed[low_end + 1] < PLANT_MAX_SPEED:
        low_end += 1
    time_below = (low_end - low_start + 1) * FRAME_SECONDS
    recovery_index = plant_index + int(round(0.5 / FRAME_SECONDS))
    speed_gain = speed[recovery_index] - plant_speed if recovery_index < len(speed) else np.nan
    reached = np.flatnonzero(speed[plant_index + 1:] >= 3.0)
    time_to_3 = (seconds[plant_index + 1 + reached[0]] - seconds[plant_index]
                 if len(reached) else np.nan)
    return {
        "braking_time_s": braking_time,
        "time_below_1_5_s": time_below,
        "speed_gain_0_5s_yd_s": speed_gain,
        "time_to_3_yd_s_s": time_to_3,
    }


def bend_event_metrics(attempt, bend_start):
    attempt = attempt.sort_values("time")
    speed = pd.to_numeric(attempt["s"], errors="coerce").to_numpy(dtype=float)
    heading = pd.to_numeric(attempt["dir"], errors="coerce").to_numpy(dtype=float)
    in_bend = attempt["in_bend"].to_numpy(dtype=bool)
    bend_end = bend_start
    while bend_end + 1 < len(in_bend) and in_bend[bend_end + 1]:
        bend_end += 1
    bend_speeds = speed[bend_start:bend_end + 1]
    entering_speed = speed[bend_start - 1] if bend_start > 0 else np.nan
    retention = (np.nanmin(bend_speeds) / entering_speed
                 if np.isfinite(entering_speed) and entering_speed > 0 else np.nan)
    yaw_rate = np.append(wrap_degrees(np.diff(heading)), 0) / FRAME_SECONDS
    bend_indices = np.arange(bend_start, bend_end + 1)
    fast_indices = bend_indices[speed[bend_indices] > 4.0]
    angular_speed = np.radians(np.abs(yaw_rate[fast_indices]))
    valid = angular_speed > 0
    radii = speed[fast_indices][valid] / angular_speed[valid]
    radius = float(np.nanmin(radii)) if len(radii) else np.nan
    return {"speed_retention_ratio": retention, "tightest_radius_yd": radius}


attempt_rows = []
for group_key, attempt in dl_tracking.groupby(EVENT_GROUP_COLUMNS, sort=False):
    attempt = attempt.sort_values("time")
    base = dict(zip(EVENT_GROUP_COLUMNS, group_key))
    for event_index in np.flatnonzero(attempt["event_plant_cut"].to_numpy()):
        attempt_rows.append({**base, "event_type": "plant_cut", "event_index": int(event_index),
            "event_side": attempt["event_side"].iloc[event_index],
            **plant_event_metrics(attempt, int(event_index)),
            "speed_retention_ratio": np.nan, "tightest_radius_yd": np.nan})
    for event_index in np.flatnonzero(attempt["event_bend"].to_numpy()):
        attempt_rows.append({**base, "event_type": "bend", "event_index": int(event_index),
            "event_side": attempt["event_side"].iloc[event_index],
            "braking_time_s": np.nan, "time_below_1_5_s": np.nan,
            "speed_gain_0_5s_yd_s": np.nan, "time_to_3_yd_s_s": np.nan,
            **bend_event_metrics(attempt, int(event_index))})

attempt_metrics = pd.DataFrame(attempt_rows)
if attempt_metrics.empty:
    raise ValueError("No plant or bend events were labeled in the selected DL drills.")
attempt_metrics["side_sign"] = attempt_metrics["event_side"].map({"L": -1, "R": 1})

metric_directions = {
    "braking_time_s": False, "time_below_1_5_s": False,
    "speed_gain_0_5s_yd_s": True, "time_to_3_yd_s_s": False,
    "speed_retention_ratio": True, "tightest_radius_yd": False,
}
for metric, higher_is_better in metric_directions.items():
    attempt_metrics[f"{metric}_percentile"] = attempt_metrics.groupby("drill_name")[metric].rank(
        pct=True, ascending=higher_is_better
    )
not_reached = attempt_metrics["event_type"].eq("plant_cut") & attempt_metrics["time_to_3_yd_s_s"].isna()
attempt_metrics.loc[not_reached, "time_to_3_yd_s_s_percentile"] = 0.0

component_metrics = {
    "cutting": ["braking_time_s_percentile", "time_below_1_5_s_percentile"],
    "explosion": ["speed_gain_0_5s_yd_s_percentile", "time_to_3_yd_s_s_percentile"],
    "bend": ["speed_retention_ratio_percentile", "tightest_radius_yd_percentile"],
}
for component, columns in component_metrics.items():
    quality = attempt_metrics[columns].mean(axis=1)
    is_component_event = attempt_metrics["event_type"].eq("bend" if component == "bend" else "plant_cut")
    attempt_metrics[f"{component}_quality"] = quality.where(is_component_event)
    attempt_metrics[f"{component}_score"] = attempt_metrics[f"{component}_quality"] * attempt_metrics["side_sign"]

name_column = "display_name"
player_columns = ["nfl_id", name_column]
id_columns = ["draft_year", "event_id", "nfl_id", "drill_name", "attempt"]
attempt_scores = attempt_metrics[id_columns + [
    "event_type", "event_index", "event_side", "braking_time_s", "time_below_1_5_s",
    "speed_gain_0_5s_yd_s", "time_to_3_yd_s_s", "speed_retention_ratio",
    "tightest_radius_yd", "cutting_score", "explosion_score", "bend_score",
]].merge(players[player_columns].drop_duplicates("nfl_id"), on="nfl_id", how="left")
player_keys = ["draft_year", "nfl_id", name_column]
score_columns = ["cutting_score", "explosion_score", "bend_score"]
player_drill_scores = attempt_scores.groupby(player_keys + ["drill_name"], dropna=False)[score_columns].mean().reset_index()
player_scores = attempt_scores.groupby(player_keys, dropna=False)[score_columns].mean().reset_index()

print(f"Scored {len(attempt_scores):,} events across {attempt_scores['nfl_id'].nunique():,} DL players.")
display(attempt_scores.head(20))
display(player_drill_scores.head(20))
display(player_scores.head(20))

Scored 1,587 events across 118 DL players.


,draft_year,event_id,nfl_id,drill_name,attempt,event_type,event_index,event_side,braking_time_s,time_below_1_5_s,speed_gain_0_5s_yd_s,time_to_3_yd_s_s,speed_retention_ratio,tightest_radius_yd,cutting_score,explosion_score,bend_score,display_name
0,2023,64065c26e3d8e72dcd199553-2023,55991,RUN_THE_HOOP_DRILL,1,bend,9,L,NaN,NaN,NaN,NaN,1.041756,5.048806,NaN,NaN,-0.468396,Isaiah McGuire
1,2023,64065c26e3d8e72dcd199553-2023,55991,RUN_THE_HOOP_DRILL,1,bend,26,R,NaN,NaN,NaN,NaN,0.594901,3.950410,NaN,NaN,0.283796,Isaiah McGuire
2,2023,64065c26e3d8e72dcd199553-2023,55991,RUN_THE_HOOP_DRILL,1,bend,39,R,NaN,NaN,NaN,NaN,1.065663,5.802551,NaN,NaN,0.458805,Isaiah McGuire
3,2023,64065c26e3d8e72dcd199553-2023,55991,RUN_THE_HOOP_DRILL,1,bend,54,L,NaN,NaN,NaN,NaN,0.643975,3.264095,NaN,NaN,-0.424735,Isaiah McGuire
4,2023,64065c26e3d8e72dcd1994e6-2023,55991,PASS_RUSH_DRILL,2,bend,7,L,NaN,NaN,NaN,NaN,1.142427,5.148261,NaN,NaN,-0.404494,Isaiah McGuire
5,2023,64065c26e3d8e72dcd1994ae-2023,55991,PASS_RUSH_DRILL,1,bend,8,R,NaN,NaN,NaN,NaN,1.062898,3.131856,NaN,NaN,0.643929,Isaiah McGuire
6,2023,64065c26e3d8e72dcd199477-2023,55991,FOUR_BAG_AGILITY_DRILL,1,plant_cut,9,L,0.4,1.6,0.933333,2.2,NaN,NaN,-0.519055,-0.374238,NaN,Isaiah McGuire
7,2023,64065c26e3d8e72dcd199477-2023,55991,FOUR_BAG_AGILITY_DRILL,1,plant_cut,24,R,0.6,0.6,2.246667,0.7,NaN,NaN,0.736280,0.875762,NaN,Isaiah McGuire
8,2023,64065c26e3d8e72dcd199477-2023,55991,FOUR_BAG_AGILITY_DRILL,1,plant_cut,43,L,1.0,0.8,1.483333,4.8,NaN,NaN,-0.410061,-0.410061,NaN,Isaiah McGuire
9,2023,64065c26e3d8e72dcd199477-2023,55991,FOUR_BAG_AGILITY_DRILL,1,plant_cut,73,B,0.5,1.2,0.956667,1.8,NaN,NaN,NaN,NaN,NaN,Isaiah McGuire


,draft_year,nfl_id,display_name,drill_name,cutting_score,explosion_score,bend_score
0,2023,55867,Will Anderson,PASS_RUSH_DRILL,NaN,NaN,0.081096
1,2023,55877,Lukas Van Ness,BODY_CONTROL_DRILL,NaN,NaN,0.617095
2,2023,55877,Lukas Van Ness,FOUR_BAG_AGILITY_DRILL,0.018039,0.034807,NaN
3,2023,55877,Lukas Van Ness,FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION,0.682609,0.803623,NaN
4,2023,55877,Lukas Van Ness,PASS_RUSH_DRILL,NaN,NaN,0.002946
5,2023,55877,Lukas Van Ness,RUN_THE_HOOP_DRILL,NaN,NaN,-0.118857
6,2023,55879,Will McDonald,BODY_CONTROL_DRILL,NaN,NaN,0.527174
7,2023,55879,Will McDonald,FOUR_BAG_AGILITY_DRILL,0.088796,0.055640,NaN
8,2023,55879,Will McDonald,FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION,0.078986,0.209420,0.514027
9,2023,55879,Will McDonald,PASS_RUSH_DRILL,NaN,NaN,-0.070069


,draft_year,nfl_id,display_name,cutting_score,explosion_score,bend_score
0,2023,55867,Will Anderson,NaN,NaN,0.081096
1,2023,55877,Lukas Van Ness,0.184181,0.227011,0.021080
2,2023,55879,Will McDonald,0.083891,0.132530,0.107340
3,2023,55905,Isaiah Foskey,-0.110133,-0.083339,-0.287253
4,2023,55911,Keion White,-0.088264,-0.212688,-0.184911
5,2023,55914,Keeanu Benton,0.136052,-0.004954,-0.038484
6,2023,55918,Gervon Dexter,0.009516,0.126474,-0.278791
7,2023,55919,Tuli Tuipulotu,-0.134202,-0.053044,0.154072
8,2023,55929,Zacch Pickens,0.124002,0.023598,-0.123514
9,2023,55942,Byron Young,0.095432,0.127461,-0.016067
